# 02 · 为什么需要 RAG

> 在动手写任何代码前，先搞清楚 RAG 到底解决了什么问题、和微调/长上下文是什么关系。

**本文件覆盖知识点**：LLM 知识截止 / 私有知识无法获得 / 幻觉 / 知识更新 / Context Window 限制 / RAG vs Fine-tuning / RAG+Fine-tuning / RAG+Long Context

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. LLM 的四个先天局限

| 局限 | 表现 | RAG 的解法 |
|------|------|-----------|
| **知识截止** | 训练数据有日期，新知识回答不了 | 知识库随时更新，无需重训 |
| **私有知识不可得** | 公司文档、内部系统不在训练集里 | 把私有知识喂进向量库再检索 |
| **幻觉** | 不知道也硬编，还显得很自信 | 强制“只依据检索资料回答” |
| **上下文有限** | 一本书放不进窗口 | 检索出最相关片段，精准利用窗口 |

一句话：**LLM 的记忆是“参数里的、过时的、公开的”；RAG 给它外接了一份“可更新的、私有的、精选的”记忆。**

In [ ]:
# 心智模型：闭卷 vs 开卷
print('''
闭卷(纯LLM): 问"今天你司新发布的产品价格?" -> 不知道就编（幻觉）
开卷(RAG):   先检索到今日最新价格单 -> 照着报价（有据可查）

RAG 价值 = 给模型一本实时可翻的"参考书"。''')

# 这里不需要 API，只需要理解：RAG 不改模型，只改模型"回答前能看到什么"。
def open_book_answer(retrieved, llm):
    """示意：开卷答题 = 把检索到的资料喂给模型"""
    return llm(f"参考资料:{retrieved}\n请据此回答")


In [ ]:
# 知识点·真调说明：LLM 四大先天局限（知识截止/私有知识/幻觉） —— 同一问题，闭卷「编或拒」，开卷照手册答
print('① 闭卷：不给任何资料，直接问一个「私有 + 只在手册里」的问题')
_llm_live(
    prompt='「星云智能客服机器人」支持哪几种部署方式？私有化部署时数据放在哪里？适合什么行业？',
    system='你是“星云智能”客服 AI。公司内部产品手册你并没有真正见过，不知道就明确回答“需要查证”，绝不要编造具体细节。',
    fallback='模型的两种典型反应（重点看是哪种）：\n'
             '· 诚实版：“抱歉，我无法访问星云的产品手册，部署方式需要向销售确认。”\n'
             '· 幻觉版：“支持公有云与本地私有化，数据放在本地服务器……”——细节越具体越可疑，它并没有看过这份手册。',
    temperature=0.2,
)
print()
print('② 开卷：把“检索命中”的《星云智能客服机器人产品手册》片段喂进去，再问同一问题')
_llm_live(
    prompt='【参考资料·手册节选】产品支持公有云 SaaS 与私有化两种部署方式。公有云版本开通即可使用，'
           '按调用量计费；私有化版本部署在客户自有服务器上，数据不出内网，适合对数据安全要求较高的金融、政务等行业。\n'
           '【问题】星云智能客服机器人支持哪几种部署方式？私有化部署数据放哪、适合什么行业？',
    system='你是“星云智能”客服 AI。只依据【参考资料】回答，资料没写的一律答“资料未提及”。',
    fallback='依据资料可稳答：支持公有云 SaaS 与私有化两种部署；私有化部署在客户自有服务器、数据不出内网，'
             '适合金融、政务等对数据安全要求较高的行业。',
    temperature=0.2,
)
print()
print('同题对比：闭卷要么“编”、要么“拒”；开卷能对着手册精确作答。')
print('→ 这就是 RAG 的立足点：把“过时/公开/存在参数里的记忆”外接成“实时/私有/可检索的资料”，'
      '一次调用同时点破知识截止、私有知识不可得、幻觉三个先天局限。')

## 2. RAG vs Fine-tuning

让模型“学会新知识”有两条主流路径：

| 维度 | RAG | 微调 Fine-tuning |
|------|-----|-----------------|
| 知识放哪 | 外部向量库 | 模型参数内 |
| 改知识成本 | 改文档即生效（秒~分钟级） | 重新训练（小时~天级） |
| 答案溯源 | 能指出来自哪篇文档 | 说不清来源 |
| 成本 | 检索+生成的推理成本 | 训练成本高，推理便宜 |
| 擅长 | 事实问答、经常更新的知识库 | 风格/格式固化、压缩 prompt |

> 二者**不互斥**：业界常用“RAG 为主 + 微调为辅”。微调负责把模型的语气、结构化能力调好，RAG 负责供给最新事实。

In [ ]:
# 知识点·真调说明：RAG vs Fine-tuning —— 模型当架构师，判定两类投诉各走哪条路、为何微调救不了“过期报价”
_llm_live(
    prompt='一家 SaaS 公司给老客户做“续费咨询”助手，已经上了大模型问答。最近两类投诉：'
           '① 回复语气生硬，像机器人念说明书；② 报价常按上季度的旧价来，动不动报错。'
           '领导想“直接微调一下模型，一次性都解决”。\n请判断：\n'
           'A. 两个问题分别该用 RAG 还是微调解决？为什么微调解决不了“报价过期”？\n'
           'B. 如果只能先做一件事，先做哪个、为什么？',
    system='你是资深 LLM 应用架构师。分 A/B 两段回答，每段不超过 4 句、第 1 句先给结论；'
           '必须结合“知识放哪、更新成本、能否溯源”讲清 RAG 与微调的职责边界。',
    fallback='A. ① 语气生硬→微调（擅长固化语气/风格，不必每次检索）；② 报价过期→必须 RAG：'
             '价格若焊在参数里就永远滞后，放进外部知识库才能“改价即生效”且可溯源。\n'
             'B. 先做②：报价错是事实性错误，比语气更伤信任；事实链路稳了再用微调统一语气。\n'
             '→ 二者不互斥：RAG 供实时事实，微调管稳定风格。',
    temperature=0.2,
)
print('→ 模型给出的边界与教材一致：微调把知识写进参数（要改就重训），RAG 把知识放在外部（改文档即生效）——'
      '“经常变的事实”归 RAG，“固化风格/格式”才值得微调，两者叠加 = 业界常说的“RAG 为主 + 微调为辅”。')

## 3. RAG + 长上下文（Long Context）

新一代模型上下文已到百万 token，那么“把整本手册全塞进去，还需要 RAG 吗？”——**仍然需要**：

1. **成本**：全塞进去的 token 数与成本随文档线性爆炸；RAG 只付“最相关几十条”的钱；
2. **效果**：研究（Lost in the Middle）表明，超长上下文里**位置居中的信息最容易丢**；检索反而让模型聚焦关键片段；
3. **速度**：KV Cache 随长度增长，全塞会导致首字延迟升高。

所以业界共识：**长上下文不能替代 RAG**；但长上下文模型让 RAG 可以把“召回片段的粒度”放宽（见第 24 课 Parent-Child 里把 Parent 整段给模型）。



In [ ]:
# 知识点·真调说明：RAG + 长上下文 —— 让模型当架构师，现场回应“上下文百万了还要 RAG 吗”
_llm_live(
    prompt='业务方说：“我们新模型的上下文已有 100 万 token，把整本产品手册一次性塞进去不就行了？还要 RAG 干嘛？”'
           '请你以 RAG 架构师身份回应。要求：按 成本 / 效果 / 速度 三类各给一条硬理由，'
           '每类配一个星云客服产品的具体例子，最后用一句话说清“长上下文与 RAG”的定位关系。',
    system='你是 RAG 架构师，回应要专业、分条、结合例子，不情绪化。',
    fallback='成本：整本手册全塞，每次提问按百万 token 计费，量一大成本线性爆炸；RAG 只付“最相关几十条”的钱。'
             '例：万页手册全塞 vs 只检索专业版那几页，费用差出数量级。\n'
             '效果：超长上下文有 Lost-in-the-Middle，关键句埋在中段照样丢；检索让模型聚焦关键片段。\n'
             '速度：KV Cache 随长度增长，全塞导致首字延迟升高，问答体感变卡。\n'
             '结论：长上下文扩大“装得下”，RAG 决定“装得对、装得省”——互补而非替代。',
    temperature=0.2,
)
print('→ 长上下文解决“装得下”，RAG 解决“装得对、装得省”；所以两者互补而不是替代，'
      '并且长模型让 RAG 可以把“给模型的片段粒度”放宽（24 课 Parent-Child 的思路）。')

## 小结

- RAG 解决 **知识截止 / 私有数据 / 幻觉 / 上下文有限** 四大问题；
- 与**微调互补**、与**长上下文互补**，是生产级 LLM 应用的标配能力。

下一步看 RAG 的完整架构，理解每一层为什么存在。